### NanoGPT Implementation
- Implementation of Karpathy's lecture on building GPT from scratch: https://www.youtube.com/watch?v=kCc8FmEb1nY&list=PLAqhIrjkxbuWI23v9cThsA9GvCAUhRvKZ&index=10

- Google collab from lecture: https://colab.research.google.com/drive/1JMLa53HDuA-i7ZBmqV7ZnA3c_fvtXnx-?usp=sharing#scrollTo=O6medjfRsLD9


### Get input data

In [ ]:
# import urllib.request

# url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
# urllib.request.urlretrieve(url, "input.txt")

### Build data processing and model in pieces

In [ ]:
# read it in to inspect it
with open("input.txt", "r", encoding="utf-8") as f:
    text = f.read()

In [ ]:
print("length of dataset in characters: ", len(text))

In [ ]:
print(text[:1000])

In [ ]:
# unique characters found in text
chars = sorted(set(text))
vocab_size = len(chars)

print("".join(chars))

In [ ]:
# map chars to integers and vice versa
stoi = {st: i for i, st in enumerate(chars)}
itos = {i: st for i, st in enumerate(chars)}

# functions to encode and decode a given sequence using stoi and itos mapping
encode = lambda x: [stoi[c] for c in x]
decode = lambda x: "".join([itos[i] for i in x])

print(encode("my name is zeal"))
print(decode(encode("my name is zeal")))

In [ ]:
# train and test splits
import torch

data = torch.tensor(encode(text), dtype=torch.long)

# 90% training and 10% validation
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

print(f"Data size: Train = {len(train_data)} | Val = {len(val_data)}")

In [ ]:
# extract one block of training examples
# a single block contains block_size number of examples

block_size = 8  # same as context_length

x = train_data[:block_size]  # single block
y = train_data[1 : block_size + 1]


print("Single block of example packs block_size number of examples")
for i in range(block_size):
    context = x[: i + 1]
    target = y[i]
    print(f"Example {i} --> context = {context} and target = {target} ")


In [ ]:
# data loader
block_size = 8
batch_size = 4


# output dimension: (B,T) where B (batch dim) is batch_size, T (time dim) is block_size
def get_batch(split):
    data = train_data if split == "train" else val_data
    # randomly pick one index per batch.
    batch_idx = torch.randint(low=0, high=len(data) - block_size, size=(batch_size,))
    # extract each batch of examples that start at their respective idx and end at idx+block_size
    x = torch.stack([data[ix : ix + block_size] for ix in batch_idx])
    y = torch.stack([data[ix + 1 : ix + block_size + 1] for ix in batch_idx])
    return x, y


xb, yb = get_batch(split="train")

print(f"Inputs: {x.shape} \n {x}")
print(f"Targets: {y.shape} \n {y}")

# -------------------------------------------------------------------------------------------------
# visualize every example packed in these four batches for our DECODER transformer block
eg = 0
for batch in range(batch_size):
    for time in range(block_size):
        context = xb[batch, 0 : time + 1]
        target = yb[batch, time]
        print(f"Example {eg} --> context = {context} and target = {target}")
        eg += 1


In [ ]:
import torch
from torch import nn
from torch.nn import functional as F

torch.manual_seed(1266)

In [ ]:
# start with bigram model. Bigram model uses a (vocab_size, vocab_size) embedding matrix
# passing "x" of size (3,4) to this matrix --> bigram_embedding_matrix(x) --> it will return a (3,4,vocab_size) output
# where, each element of x is now embedded into vocab_size dimensions


class BigramModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, x, targets=None):
        # x is of size (B,T), where each element is a character index from the vocabulary
        # target is of size (B,T)

        logits = self.token_embedding_table(x)  # (B,T,vocab_size) after passing through the embedding table

        # loss
        if targets == None:
            loss = None
        else:
            # pytorch requires shape (B,C,T) instead of (B,T,C), which is confusing, so we combine the first two dims.
            B, T, C = logits.shape
            logits = logits.view(B * T, C)
            targets = targets.view(B * T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, x, max_new_tokens):
        # generate one token at a time
        for i in range(max_new_tokens):
            # forward pass through the model
            logits, _ = self.forward(x)  # logits is (B,T,vocab_size)
            # focus only on the last timestep; no dependence on past here
            logits = logits[:, -1, :]  # (B,vocab_size)
            # convert logits to probabilities
            probs = F.softmax(logits, dim=-1)
            # sample next token from probability distribution
            next_token = torch.multinomial(probs, num_samples=1)  # (B,1)
            # append next_token to the original context
            x = torch.cat((x, next_token), dim=1)  # (B,T+1)
        return x


# call the model
model = BigramModel()
logits, loss = model(x=xb, targets=yb)
print(f"Loss = {loss}")

model_output = model.generate(x=torch.zeros((1, 1), dtype=torch.long), max_new_tokens=1000)
print(decode(model_output[0].tolist()))

In [ ]:
# train the bigram model

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

for steps in range(1000):
    xb, yb = get_batch("train")

    _, loss = model(x=xb, targets=yb)
    # set all gradients None
    optimizer.zero_grad(set_to_none=True)
    # run backward pass
    loss.backward()
    # make gradient update for each parameter
    optimizer.step()

print(loss.item())


In [ ]:
model_output = model.generate(x=torch.zeros((1, 1), dtype=torch.long), max_new_tokens=1000)
print(decode(model_output[0].tolist()))

### Adding positional encoding and single self-attention head with ffn

In [ ]:
import torch
from torch import nn
from torch.nn import functional as F

torch.manual_seed(1266)

# ---------------------------Get data ready---------------------------
# read input data
with open("input.txt", "r", encoding="utf-8") as f:
    text = f.read()

# unique characters found in text
chars = sorted(set(text))
vocab_size = len(chars)

# map chars to integers and vice versa
stoi = {st: i for i, st in enumerate(chars)}
itos = {i: st for i, st in enumerate(chars)}
# functions to encode and decode a given sequence using stoi and itos mapping
encode = lambda x: [stoi[c] for c in x]
decode = lambda x: "".join([itos[i] for i in x])

# train-val split: 90% training and 10% validation
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

# ---------------------------hyperparameters---------------------------
# hyperparameters
block_size = 8
batch_size = 4
lr = 1e-3
n_embd = 64
head_size = 8
device = "cuda" if torch.cuda.is_available() else "cpu"
max_iters = 10000
# ---------------------------------------------------------------------


# data loader
# output dimension: (B,T) where B (batch dim) is batch_size, T (time dim) is block_size
def get_batch(split):
    data = train_data if split == "train" else val_data
    # randomly pick one index per batch.
    batch_idx = torch.randint(low=0, high=len(data) - block_size, size=(batch_size,))
    # extract each batch of examples that start at their respective idx and end at idx+block_size
    x = torch.stack([data[ix : ix + block_size] for ix in batch_idx])
    y = torch.stack([data[ix + 1 : ix + block_size + 1] for ix in batch_idx])
    x, y = x.to(device), y.to(device)
    return x, y


# add single self attention head to bigram model
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        # tril is pre-computed for a head as a fixed-size lower triangular matrix of size (block_size, block_size)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B, T, C = x.shape  # in this case, C = head_size
        q = self.query(x)  # (B,T,C)
        k = self.key(x)  # (B,T,C)
        v = self.value(x)  # (B,T,C)

        # calculate scaled weight matrix
        wei = q @ k.transpose(-2, -1) * C**-0.5  # (B,T,T)
        # masking the weight matrix
        # during training or inference, your input batch might have a sequence length T
        # that is shorter than block_size (for example, T=16 while block_size = 1024)
        # hence we use self.tril[:T,:T]==0 and not self.tril==0
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))  # (B,T,T)
        # softmax
        wei = F.softmax(wei, dim=-1)  # (B,T,T)

        # compute output
        out = wei @ v  # (B,T,C) i.e. (B,T,head_size) will be outputted by a single head
        return out


# main model class
class BigramModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        # self attention head
        self.sa_head = Head(head_size=head_size)
        self.lm_head = nn.Linear(head_size, vocab_size)  # projection layer from head_size -> vocab_size

    def forward(self, x, targets=None):
        B, T = x.shape
        # information and positional encoding of input are computed and added
        tok_emb = self.token_embedding_table(x)  # (B,T,n_embd)
        pos_emb = self.position_embedding_table(torch.arange(T, device=x.device))  # (T,n_embd)
        x = tok_emb + pos_emb  # (B,T,n_embd)

        # single self-attention head
        x = self.sa_head(x)  # (B,T,head_size)
        # FFN
        # why FFN here: if we pass head's output directly to cross-entropy it will error out because
        # head's output is in range 0 to head_size, where as target is in range 0 to vocab_size.
        logits = self.lm_head(x)  # (B,T,vocab_size)

        # loss
        if targets == None:
            loss = None
        else:
            # pytorch requires shape (B,C,T) instead of (B,T,C), which is confusing, so we combine the first two dims.
            B, T, C = logits.shape
            logits = logits.view(B * T, C)
            targets = targets.view(B * T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, x, max_new_tokens):
        # generate one token at a time
        for _ in range(max_new_tokens):
            # limit context to latest block_size tokens
            x_cond = x[:, -block_size:]
            # forward pass this temporary slice through the model
            logits, _ = self.forward(x_cond)  # logits is (B,T,vocab_size)
            # focus only on the last timestep; no dependence on past here
            logits = logits[:, -1, :]  # (B,vocab_size)
            # convert logits to probabilities
            probs = F.softmax(logits, dim=-1)
            # sample next token from probability distribution
            next_token = torch.multinomial(probs, num_samples=1)  # (B,1)
            # append next_token to the original context
            x = torch.cat((x, next_token), dim=1)  # (B,T+1)
        return x


# call the model
xb, yb = get_batch(split="train")

model = BigramModel()
m = model.to(device)
# number of parameters in the model
print(sum(p.numel() for p in m.parameters()) / 1e6, "M parameters")

## single forward pass
# logits, loss = m(x=xb, targets=yb)
# print(f"Loss = {loss}")

## model generation check
# model_output = model.generate(x=torch.zeros((1, 1), dtype=torch.long), max_new_tokens=1000)
# print(decode(model_output[0].tolist()))

# -------------------------------train the model-------------------------------
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

for step in range(max_iters):
    xb, yb = get_batch("train")

    _, loss = model(x=xb, targets=yb)
    # set all gradients None
    optimizer.zero_grad(set_to_none=True)
    # run backward pass
    loss.backward()
    # make gradient update for each parameter
    optimizer.step()

    # print loss every few iterations
    if step % 1000 == 0:
        print(f"Iter: {step} | Loss: {loss.item()}")

print(f"Final Loss: {loss.item()}")

context = torch.zeros((1, 1), dtype=torch.long, device=device)
# model_output = m.generate(x=context, max_new_tokens=2000)
# print(decode(model_output[0].tolist()))
print(decode(m.generate(context, max_new_tokens=2000)[0].tolist()))


0.006793 M parameters
Iter: 0 | Loss: 4.097550868988037
Iter: 1000 | Loss: 2.8182942867279053
Iter: 2000 | Loss: 2.247951030731201
Iter: 3000 | Loss: 2.335516929626465
Iter: 4000 | Loss: 2.5938055515289307
Iter: 5000 | Loss: 2.5994932651519775
Iter: 6000 | Loss: 2.7757251262664795
Iter: 7000 | Loss: 2.3520689010620117
Iter: 8000 | Loss: 2.3567793369293213
Iter: 9000 | Loss: 2.227532386779785
Final Loss: 2.3101885318756104


I
HULUSL.

Who osgtris jundlle the-r weivive nol fath sneg if sifour:
Fomy orecavebre, my.

USHESA:
Bus whamoreent, slt prind yonge pth bied Igr sy fbar, ngrt otullll hmamy, bere alg vacirlt ofid Thyist hiupeme ofrees?

Bed that nfrighe, me'led stiend maxleath bekent ere; sounk er:
Corot,, tiln cous fe'pre wt hame ow
Mine ot ther yom oul the nn waven th,
Ctass? feat,
Y.
SE:
Go
IRjvecd fe to bre wet wae
Mck tn.
C3y mef rot at:
II kmft furalgns:
Tof a, rchee der.
NDUSEE:
O
S' Nlee I oth

Awo
I has 'net dlot tt tot woud o! blre fr
Wheillor rit wont weras,
I SHe sy ad, 

### Adding multi-attention head, blocks, skip connections, layernorm, dropout